In [ ]:
import polars as pl
import os, tempfile, itertools
from aeon.datasets.tsc_datasets import univariate
from tscglue import utils
import numpy as np

# Load cached parquet files
cache_dir = os.path.join(tempfile.gettempdir(), "tsc-glue-cache")
local_paths = sorted(os.path.join(cache_dir, f) for f in os.listdir(cache_dir) if f.endswith(".parquet"))
res_mine = pl.read_parquet(local_paths)

In [ ]:
model = "loky-stacker-v10-base-r3"
df = res_mine.filter(pl.col("model") == model)

# Build expected combos (112 datasets x 30 folds)
all_datasets = sorted(univariate)
all_folds = list(range(30))
expected = pl.DataFrame({
    "dataset": [d for d, f in itertools.product(all_datasets, all_folds)],
    "fold": [f for d, f in itertools.product(all_datasets, all_folds)],
})

# Find missing
actual = df.select(["dataset", "fold"]).unique()
missing = expected.join(actual, on=["dataset", "fold"], how="anti")

print(f"Total expected: {len(expected)}")
print(f"Total actual:   {len(actual)}")
print(f"Total missing:  {len(missing)}")

Total expected: 3840
Total actual:   3160
Total missing:  680


In [ ]:
# Which datasets have missing folds, and which folds are missing?
by_dataset = (
    missing.group_by("dataset")
    .agg(
        pl.col("fold").sort().alias("missing_folds"),
        pl.col("fold").count().alias("n_missing"),
    )
    .sort("n_missing", descending=True)
)
print("Datasets with missing folds:")
print(by_dataset)

Datasets with missing folds:
shape: (102, 3)
┌───────────────────────────────┬───────────────┬───────────┐
│ dataset                       ┆ missing_folds ┆ n_missing │
│ ---                           ┆ ---           ┆ ---       │
│ str                           ┆ list[i64]     ┆ u32       │
╞═══════════════════════════════╪═══════════════╪═══════════╡
│ AllGestureWiimoteZ            ┆ [0, 1, … 29]  ┆ 30        │
│ DodgerLoopDay                 ┆ [0, 1, … 29]  ┆ 30        │
│ GesturePebbleZ1               ┆ [0, 1, … 29]  ┆ 30        │
│ GestureMidAirD3               ┆ [0, 1, … 29]  ┆ 30        │
│ GestureMidAirD2               ┆ [0, 1, … 29]  ┆ 30        │
│ …                             ┆ …             ┆ …         │
│ DistalPhalanxOutlineAgeGroup  ┆ [18]          ┆ 1         │
│ ProximalPhalanxOutlineCorrect ┆ [15]          ┆ 1         │
│ RefrigerationDevices          ┆ [16]          ┆ 1         │
│ ShapesAll                     ┆ [1]           ┆ 1         │
│ OSULeaf                

In [ ]:
# Which folds fail the most?
by_fold = (
    missing.group_by("fold")
    .agg(pl.col("dataset").count().alias("n_missing_datasets"))
    .sort("fold")
)
print("Missing count by fold:")
print(by_fold)

Missing count by fold:
shape: (30, 2)
┌──────┬────────────────────┐
│ fold ┆ n_missing_datasets │
│ ---  ┆ ---                │
│ i64  ┆ u32                │
╞══════╪════════════════════╡
│ 0    ┆ 16                 │
│ 1    ┆ 28                 │
│ 2    ┆ 17                 │
│ 3    ┆ 16                 │
│ 4    ┆ 17                 │
│ …    ┆ …                  │
│ 25   ┆ 30                 │
│ 26   ┆ 17                 │
│ 27   ┆ 17                 │
│ 28   ┆ 17                 │
│ 29   ┆ 17                 │
└──────┴────────────────────┘


In [ ]:
# Dataset stats for missing datasets - look for patterns (size, length, n_classes)
missing_dataset_names = by_dataset["dataset"].to_list()

stats = []
for dataset in missing_dataset_names:
    X_train, y_train, X_test, y_test = utils.load_dataset(dataset)
    stats.append({
        "dataset": dataset,
        "n_train": X_train.shape[0],
        "n_test": X_test.shape[0],
        "n_classes": len(np.unique(y_train)),
        "series_length": X_train.shape[2],
        "n_missing_folds": by_dataset.filter(pl.col("dataset") == dataset)["n_missing"][0],
    })

stats_df = pl.DataFrame(stats).sort("n_missing_folds", descending=True)
print("Stats of datasets with missing folds:")
stats_df

Stats of datasets with missing folds:


dataset,n_train,n_test,n_classes,series_length,n_missing_folds
str,i64,i64,i64,i64,i64
"""AllGestureWiimoteZ""",300,700,10,500,30
"""DodgerLoopDay""",67,77,7,288,30
"""GesturePebbleZ1""",132,172,6,455,30
"""GestureMidAirD3""",208,130,26,360,30
"""GestureMidAirD2""",208,130,26,360,30
…,…,…,…,…,…
"""DistalPhalanxOutlineAgeGroup""",400,139,3,80,1
"""ProximalPhalanxOutlineCorrect""",600,291,2,80,1
"""RefrigerationDevices""",375,375,3,720,1


In [ ]:
# Full list of missing (dataset, fold) pairs for easy copy-paste debugging
print("All missing (dataset, fold) pairs:")
missing.sort(["dataset", "fold"])

All missing (dataset, fold) pairs:


dataset,fold
str,i64
"""AllGestureWiimoteX""",0
"""AllGestureWiimoteX""",1
"""AllGestureWiimoteX""",2
"""AllGestureWiimoteX""",3
"""AllGestureWiimoteX""",4
…,…
"""Wafer""",17
"""Wafer""",18
"""Wine""",10


In [ ]:
missing.group_by('dataset').count()

/tmp/ipykernel_342692/617116991.py:1: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  missing.group_by('dataset').count()


dataset,count
str,u32
"""HandOutlines""",6
"""BeetleFly""",3
"""GunPointMaleVersusFemale""",2
"""GestureMidAirD1""",30
"""ECG200""",1
…,…
"""GestureMidAirD2""",30
"""OliveOil""",3
"""DistalPhalanxTW""",2
